In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Better Feature Engineering
for df in [train, test]:
    # Fix missing values
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    
    # Encode
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    
    # New features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df["Title"] = df["Title"].map({
        "Mr": 0, "Miss": 1, "Mrs": 2, 
        "Master": 3, "Dr": 4
    }).fillna(5)

# Features
features = ["Pclass", "Sex", "Age", "Fare", 
            "FamilySize", "IsAlone", "Title"]
X = train[features]
y = train["Survived"]
X_test = test[features]
# Back to best settings + add Title
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42)

features = ["Pclass", "Sex", "Age", "Fare", 
            "FamilySize", "IsAlone", "Title",
            "SibSp", "Parch"]

X = train[features]
X_test = test[features]

model.fit(X, y)

predictions = model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done!")
# # Train
# model = RandomForestClassifier(
#     n_estimators=200,
#     max_depth=6,
#     random_state=42)
# model.fit(X, y)

# # Submit
# predictions = model.predict(X_test)
# output = pd.DataFrame({
#     'PassengerId': test.PassengerId, 
#     'Survived': predictions
# })
# output.to_csv('submission.csv', index=False)
# print("Done! Submit this file!")

Done!


In [3]:
# print(train["Title"].value_counts())
# print(train[["Name", "Title"]].head(10))

Title
0.0    517
1.0    182
2.0    125
3.0     40
5.0     20
4.0      7
Name: count, dtype: int64
                                                Name  Title
0                            Braund, Mr. Owen Harris    0.0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    2.0
2                             Heikkinen, Miss. Laina    1.0
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    2.0
4                           Allen, Mr. William Henry    0.0
5                                   Moran, Mr. James    0.0
6                            McCarthy, Mr. Timothy J    0.0
7                     Palsson, Master. Gosta Leonard    3.0
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)    2.0
9                Nasser, Mrs. Nicholas (Adele Achem)    2.0


In [4]:
# # Better tuned model
# model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=7,
#     min_samples_split=10,
#     min_samples_leaf=5,
#     random_state=42)
# model.fit(X, y)

# # Submit
# predictions = model.predict(X_test)
# output = pd.DataFrame({
#     'PassengerId': test.PassengerId,
#     'Survived': predictions
# })
# output.to_csv('submission.csv', index=False)
# print("Done!")

Done!


In [6]:
# Use exact same as V3 but add Title only
features = ["Pclass", "Sex", "SibSp", 
            "Parch", "Fare", "Title"]

# Make sure Embarked is encoded
for df in [train, test]:
    embarked = pd.get_dummies(df["Embarked"], 
                              prefix="Embarked").astype(int)

X = pd.get_dummies(train[["Pclass", "Sex", 
                           "SibSp", "Parch", "Fare"]])
X["Title"] = train["Title"]

X_test = pd.get_dummies(test[["Pclass", "Sex", 
                               "SibSp", "Parch", "Fare"]])
X_test["Title"] = test["Title"]

model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    random_state=1)
model.fit(X, y)

predictions = model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done!")

Done!
